In [1]:
import numpy as np
import sys
import os
from scipy.io import savemat, loadmat
from scipy import interpolate
from natsort import natsorted
import h5py
import pickle
import torch
from torchvision import transforms
from dflat.render import hsi_to_rgb, general_convolve

# Resave the Raw HSI Cubes

In [19]:
def interpolate_HS_Cube(new_channels_nm, hs_cube, hs_bands):
    # Throw an error if we try to extrapolate
    if (min(new_channels_nm) < min(hs_bands) - 1) or (
        max(new_channels_nm) > max(hs_bands) + 1
    ):
        raise ValueError(
            f"Extrapoaltion outside of measurement data is not allowed: {min(hs_bands)}-{max(hs_bands)}"
        )

    interpfun = interpolate.interp1d(
        hs_bands,
        hs_cube,
        axis=-1,
        kind="linear",
        assume_sorted=True,
        fill_value="extrapolate",
        bounds_error=False,
    )
    resampled = interpfun(new_channels_nm)
    return resampled

In [ ]:
## First we want to reprocess the data and resave all the data files as .mat files 
# To reduce memory, keep only spectral channels we want later --> 400 to 700 10 nm
# use scipy savemat which can do float32 instead of float16
def read_envi_header(hdr_file):
    header_info = {}
    wavelengths = []

    with open(hdr_file, 'r') as f:
        for line in f:
            if '=' in line:
                key, value = line.strip().split('=', 1)
                key = key.strip().lower()
                value = value.strip().strip('{}').strip()
                header_info[key] = value

            if 'wavelength' in line.lower():
                while True:
                    line = f.readline().strip()
                    if '}' in line:
                        break
                    values = line.split(',')
                    wavelengths += [float(w) for w in values if w.strip()]
                header_info['wavelength'] = wavelengths

            if 'fwhm' in line.lower():
                fwhms = []
                while True:
                    line = f.readline().strip()
                    if '}' in line:
                        break
                    values = line.split(',')
                    fwhms += [float(f) for f in values if f.strip()]
                header_info['fwhm'] = fwhms

    header_info['samples'] = int(header_info['samples'])
    header_info['lines'] = int(header_info['lines'])
    header_info['bands'] = int(header_info['bands'])
    header_info['data type'] = int(header_info['data type'])
    header_info['header offset'] = int(header_info['header offset'])
    header_info['byte order'] = int(header_info['byte order'])

    return header_info, wavelengths

def load_envi_raw_data(raw_file, header_info):
    dtype_map = {
        12: np.uint16,  # Adjust this based on your specific data type
        # Add other data types as needed based on ENVI specification
    }
    dtype = dtype_map.get(header_info['data type'])
    if dtype is None:
        raise ValueError(f"Unsupported data type: {header_info['data type']}")

    shape = (header_info['lines'], header_info['samples'], header_info['bands'])

    raw_data = np.fromfile(raw_file, dtype=dtype)

    # Check if byte order needs to be swapped
    if (header_info['byte order'] == '1' and sys.byteorder == 'little') or \
       (header_info['byte order'] == '0' and sys.byteorder == 'big'):
        raw_data = raw_data.byteswap()

    # Handle interleave format
    if header_info['interleave'] == 'bil':
        raw_data = raw_data.reshape((shape[0], shape[2], shape[1]))  # Lines, Bands, Samples
        raw_data = np.transpose(raw_data, (0, 2, 1))  # Convert to Lines, Samples, Bands
    elif header_info['interleave'] == 'bip':
        raw_data = raw_data.reshape((shape[0], shape[1], shape[2]))  # Lines, Samples, Bands
    elif header_info['interleave'] == 'bsq':
        raw_data = raw_data.reshape(shape)  # Lines, Samples, Bands
    else:
        raise ValueError(f"Unsupported interleave format: {header_info['interleave']}")

    # Transpose the image to correct the orientation
    return raw_data.transpose(1,0,2)

data_fold = "/media/deanhazineh/T7/ICVL Dataset/"
output_dir = "/home/deanhazineh/ssd4tb_mounted/DiffVis/scripts/datasets/ICVL_repackaged_400_700/"
ids = np.arange(0, 201, 1)
interp_ch = np.arange(400, 710, 10)

for id in ids:
    mat_filename = os.path.join(output_dir, f'{id}.mat')
    if os.path.exists(mat_filename):
        print("skipping", id)
        continue
    print(mat_filename)
    
    hdr_file = os.path.join(data_fold, "hdr", f"{id}.hdr")
    header_info, ch = read_envi_header(hdr_file)
    
    raw_file = os.path.join(data_fold, f"{id} Yi-Chun Hung.raw")
    raw_data = load_envi_raw_data(raw_file, header_info).astype(np.float32)
    raw_data = raw_data / raw_data.max()
    raw_data = interpolate_HS_Cube(interp_ch, raw_data, ch)
    savemat(mat_filename, {"hsi": raw_data}, )


In [26]:
# Save a version of the HSI cubes in a common range of 420 to 700 with 64 chunking (float16)
datpath = "/home/deanhazineh/ssd4tb_mounted/DiffVis/scripts/datasets/ICVL_repackaged_400_700/"
savepath = f"/home/deanhazineh/ssd4tb_mounted/DiffVis/scripts/datasets/HSI_multiset_420_700/"
fnames = natsorted(os.listdir(datpath))

ch = np.linspace(400, 700, 31)
interp_ch = np.arange(420, 710, 10)
chunks = 64

for f in fnames:
    datfile = os.path.join(datpath, f)
    hsi = loadmat(datfile)["hsi"].astype(np.float32)
    hsi = hsi/hsi.max()
    
    hsi = np.clip(interpolate_HS_Cube(interp_ch, hsi, ch),0,1).astype(np.float16)
    hdf5_file_path = savepath+f'ICVL_{os.path.splitext(f)[0]}.h5'
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('hsi', data=hsi, chunks=(chunks, chunks, 28), dtype='float16')

(1392, 1304, 31)
(1392, 1306, 31)
(1392, 1304, 31)
(1392, 1308, 31)
(1392, 1304, 31)
(1392, 1308, 31)
(1392, 1306, 31)
(1392, 1304, 31)
(1392, 1303, 31)
(1392, 1305, 31)
(1392, 1303, 31)
(1392, 1306, 31)
(1392, 1301, 31)
(1392, 1308, 31)
(1392, 1305, 31)
(1392, 1306, 31)
(1392, 1305, 31)
(1392, 1306, 31)
(1392, 1306, 31)
(1392, 1306, 31)
(1392, 1306, 31)
(1392, 1306, 31)
(1392, 1307, 31)
(1392, 1083, 31)
(1392, 1306, 31)
(1392, 1082, 31)
(1392, 1304, 31)
(1392, 1306, 31)
(1392, 1303, 31)
(1392, 1305, 31)
(1392, 1302, 31)
(1392, 1305, 31)
(1392, 1305, 31)
(1392, 1301, 31)
(1392, 1303, 31)
(1392, 1305, 31)
(1392, 1305, 31)
(1392, 1304, 31)
(1392, 1305, 31)
(1392, 1303, 31)
(1392, 1305, 31)
(1392, 1306, 31)
(1392, 1306, 31)
(1392, 1306, 31)
(1392, 1306, 31)
(1392, 1306, 31)
(1392, 1305, 31)
(1392, 1202, 31)
(1392, 1196, 31)
(1392, 1194, 31)
(1392, 1191, 31)
(1392, 1083, 31)
(1392, 1083, 31)
(1392, 1307, 31)
(1392, 1305, 31)
(1392, 1303, 31)
(1392, 1303, 31)
(1392, 1304, 31)
(1392, 1305, 3

# Prerender the measurements

In [2]:
aug = True
psf_path = "./metasurfaces/L4v2_lens_psf_compact32.pickle"
datpath = "./datasets/ICVL_repackaged_400_700/"
savepath = "./datasets/ICVL_prerendered/"
os.makedirs(savepath, exist_ok=True)
skip = [94, 95, 97, 98, 102, 105, 157, 187, 189]
test = [8, 27, 43, 43, 46, 90, 103, 150, 153, 166]

with open(psf_path, "rb") as file:
    data = pickle.load(file)
    psf = data["psf_int"]
    psf = psf / np.max(psf)
    wl = data["wl"]
psf = torch.tensor(psf / np.max(psf), dtype=torch.float32).to(device="cuda")

transform = transforms.Compose([transforms.ToTensor()])
chunks = 64
dtype='float32'

def render_and_save(hsi, savepath, f, auglabel):
    mhsi = general_convolve(hsi, psf, rfft=True)
    mgs = torch.sum(mhsi, dim=0, keepdim=True)
    mgs = mgs / torch.max(mgs)
    #mrgb = hsi_to_rgb(mhsi, wl, tensor_ordering=True, projection="Basler_Bayer", normalize=True)
    #rgb = hsi_to_rgb(hsi, wl, tensor_ordering=True, projection="Basler_Bayer", normalize=True)
    
    hsi = hsi.permute(1,2,0).cpu().numpy()
    mgs = mgs.permute(1,2,0).cpu().numpy()
    #mrgb = mrgb.permute(1,2,0).cpu().numpy()
    #rgb = rgb.permute(1,2,0).cpu().numpy()

    hdf5_file_path = os.path.join(savepath, f'{os.path.splitext(f)[0]}{auglabel}.h5')
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('hsi', data=hsi, chunks=(chunks, chunks, 31), dtype=dtype)
        hf.create_dataset('mgs', data=mgs, chunks=(chunks, chunks, 1), dtype=dtype)
        #hf.create_dataset('mrgb', data=mrgb, chunks=(chunks, chunks, 3), dtype=dtype)
        #hf.create_dataset('rgb', data=rgb, chunks=(chunks, chunks, 3), dtype=dtype)
    return

fnames = natsorted(os.listdir(datpath))
for it, f in enumerate(fnames):
    id = int(os.path.splitext(f)[0])
    if id in skip:
        continue

    datfile = os.path.join(datpath, f)
    hsi = loadmat(datfile)["hsi"]
    hsi = transform(hsi).to('cuda')
    hsi = hsi / hsi.max()
    f = "ICVL_train_" + str(id) if id not in test else "ICVL_test_"+str(id)
    render_and_save(hsi, savepath, f, "")
    if aug: 
        render_and_save(hsi.flip(dims=[2]), savepath, f, "_hflip")
        render_and_save(hsi.flip(dims=[1]), savepath, f, "_vflip")
        render_and_save(hsi.flip(dims=[1,2]), savepath, f, "_vhflip")

/home/deanhazineh/anaconda3/envs/diffvis/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/deanhazineh/anaconda3/envs/diffvis/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
